# Kia Sportage reserved measurements
Speed-blind detection, geometry, masks and optical flow. No speed errors are calculated here. Text annotations are not uploaded. Pinned NAVER model retains its CC BY-NC-SA 3.0 terms.

In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys
sources={'validation_inference.py': '"""Speed-blind detection and cuboid extraction for constant-speed validation clips."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport time\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\n\nMODEL_REPO=\'mehmetkeremturkcan/UrbanOmniDetect\'\nMODEL_REVISION=\'2701a2eec9a73afb5ffd852740d125dfcd065ce5\'\nMODEL_FILE=\'checkpoints/urbanomnidetect_yolo11x-p2_1920.pt\'\nMODEL_SHA=\'9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d\'\n\n\ndef overlaps(a,b):\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    intersection=np.prod(np.maximum(0,hi-lo),axis=2)\n    return intersection/(np.prod(a[:,2:]-a[:,:2],axis=1)[:,None]+\n                         np.prod(b[:,2:]-b[:,:2],axis=1)[None,:]-intersection+1e-9)\n\n\ndef infer(folder,output,device=\'0\'):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    model_dir=output.parent/\'models\';model_dir.mkdir(exist_ok=True)\n    weights=model_dir/\'urban_geometry.pt\'\n    if not weights.exists():\n        urllib.request.urlretrieve(f\'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{MODEL_FILE}\',weights)\n    assert hashlib.sha256(weights.read_bytes()).hexdigest()==MODEL_SHA\n    geometry_model=YOLO(str(weights))\n    for video in sorted(Path(folder).rglob(\'*.MP4\')):\n        target=output/video.stem;target.mkdir(exist_ok=True)\n        tracker=YOLO(str(model_dir/\'yolo11s.pt\'))\n        cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n        width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps=float(cap.get(cv2.CAP_PROP_FPS));total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        observations=[];tracks=[];times=[];start=time.monotonic();index=0\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            pts=float(cap.get(cv2.CAP_PROP_POS_MSEC))/1000\n            timestamp=pts if abs(pts-index/fps)<.1 else index/fps\n            times.append(timestamp)\n            if index==0:cv2.imwrite(str(target/\'reference.jpg\'),frame)\n            detected=tracker.track(frame,persist=True,tracker=\'bytetrack.yaml\',device=device,\n                                   imgsz=1280,classes=[2,5,7],conf=.15,verbose=False)[0]\n            boxes=detected.boxes.xyxy.cpu().numpy()\n            ids=detected.boxes.id.cpu().numpy().astype(int) if detected.boxes.id is not None else np.full(len(boxes),-1)\n            scores=detected.boxes.conf.cpu().numpy()\n            for box,tid,score in zip(boxes,ids,scores):\n                tracks.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(tid),\n                               \'box\':box.tolist(),\'confidence\':float(score)})\n            selected=[i for i,box in enumerate(boxes) if box[3]-box[1]>=45 and ids[i]>=0]\n            if selected:\n                result=geometry_model.predict(frame,device=device,imgsz=1920,classes=[0],conf=.15,verbose=False)[0]\n                if result.keypoints is not None and len(result.boxes):\n                    gb=result.boxes.xyxy.cpu().numpy();gs=result.boxes.conf.cpu().numpy()\n                    kp=result.keypoints.data.cpu().numpy()\n                    overlap=overlaps(boxes[selected],gb);ii,jj=linear_sum_assignment(1-overlap)\n                    for i,j in zip(ii,jj):\n                        if overlap[i,j]<.25:continue\n                        k=selected[i]\n                        observations.append({\'frame_index\':index,\'time_s\':timestamp,\'track_id\':int(ids[k]),\n                            \'baseline_confidence\':float(scores[k]),\'geometry_confidence\':float(gs[j]),\n                            \'association_iou\':float(overlap[i,j]),\'baseline_box\':boxes[k].tolist(),\n                            \'geometry_box\':gb[j].tolist(),\'keypoints\':kp[j].tolist()})\n                    if len(observations)%20<2:\n                        cv2.imwrite(str(target/f\'geometry_{index:04d}.jpg\'),result.plot())\n            index+=1\n        cap.release()\n        assert index==total\n        (target/\'observations.json\').write_text(json.dumps(observations),encoding=\'utf-8\')\n        (target/\'tracks.json\').write_text(json.dumps(tracks),encoding=\'utf-8\')\n        if observations:\n            points=np.array([r[\'keypoints\'] for r in observations])\n            vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n            ground_indices=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n            agreement=float(max(vote.mean(),1-vote.mean()))\n        else:ground_indices=[];agreement=0.0\n        report={\'source_video\':video.name,\'width\':width,\'height\':height,\'fps\':fps,\'frames\':total,\n                \'duration_s\':total/fps,\'timestamps\':times,\'model_sha256\':MODEL_SHA,\n                \'model_revision\':MODEL_REVISION,\'model_filename\':MODEL_FILE,\'ground_indices\':ground_indices,\n                \'ground_order_vote_agreement\':agreement,\'runtime_s\':time.monotonic()-start,\n                \'observations\':len(observations),\'reference_speeds_used_by_inference\':False}\n        (target/\'geometry_run_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(video.name,\'frames\',index,\'geometry\',len(observations),\'seconds\',round(report[\'runtime_s\'],1),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--device\',default=\'0\')\n    args=parser.parse_args();infer(args.input,args.output,args.device)\n', 'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False, legacy_normalization=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    if legacy_normalization:\n        normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    q_starts=([0,1,2] if not fast else [1,2]) if legacy_normalization else ([0,2,5,10,20] if not fast else [2,10])\n    aspect_starts=([0,0.7] if not fast else [0]) if legacy_normalization else ([-3,0] if not fast else [-2])\n    starts=[(0,q,a,0) for q in q_starts for a in aspect_starts]\n    fits=[]\n    for theta in starts:\n        bound=4 if legacy_normalization else 100\n        shape_bound=2.5 if legacy_normalization else 7\n        shear_bound=4 if legacy_normalization else 30\n        result=least_squares(residual,theta,bounds=([-bound,-bound,-shape_bound,-shear_bound],[bound,bound,shape_bound,shear_bound]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=500)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height, longitudinal_order=\'checkpoint\'):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            if longitudinal_order==\'motion\':\n                quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            if longitudinal_order==\'motion\':\n                lo=max(0,j-3);hi=min(len(group)-1,j+3)\n                motion=centers[hi]-centers[lo]\n                if np.linalg.norm(motion)<5:continue\n                edge=np.roll(quad,-1,axis=0)-quad\n                alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n                scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n                if abs(scores[0]-scores[1])<0.08:continue\n                if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            elif longitudinal_order!=\'checkpoint\':\n                raise ValueError(\'Unknown longitudinal corner-order method.\')\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'estimated_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'benchmark_speed.py': '"""Label-independent single-pass estimators; reference speeds are used only by evaluation/tuning."""\nfrom collections import defaultdict\nfrom pathlib import Path\nimport json\nimport math\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nimport calibrate_speed as geometry\n\n\ndef target_records(observations, report, min_height=0, max_height=10000, corner_order=\'checkpoint\'):\n    prepared=geometry.prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'],corner_order)\n    boxes={(r[\'track_id\'],r[\'frame_index\']):r[\'baseline_box\'] for r in observations}\n    groups=defaultdict(list)\n    for row in prepared:\n        box=boxes[row[\'track_id\'],row[\'frame_index\']]\n        if min_height<=box[3]-box[1]<=max_height:groups[row[\'track_id\']].append(row)\n    if not groups:return []\n    # The moving pass with the largest observed footprint is the target, independent of speed labels.\n    target=max(groups.values(),key=lambda rows:max(abs(cv2.contourArea(np.array(r[\'quad\'],np.float32))) for r in rows))\n    return sorted(target,key=lambda r:r[\'time_s\'])\n\n\ndef estimate_single_pass(observations, report, config):\n    rows=target_records(observations,report,config.get(\'min_height\',0),config.get(\'max_height\',10000),config.get(\'corner_order\',\'checkpoint\'))\n    if len(rows)<8 or rows[-1][\'time_s\']-rows[0][\'time_s\']<.25:\n        return {\'available\':False,\'reason\':\'insufficient_unclipped_ground_geometry\',\'observations\':len(rows)}\n    length=config.get(\'length_m\',4.5);width=config.get(\'width_m\',1.8)\n    H,diagnostics=geometry.fit_mapping(rows,report[\'width\'],report[\'height\'],length,width,\n                                      legacy_normalization=config.get(\'legacy_normalization\',False))\n    calculated=geometry.estimate_track(rows,H)\n    result={\'available\':True,\'raw_kmh\':calculated[\'experimental_kmh\'],\n            \'estimated_kmh\':calculated[\'experimental_kmh\']*config.get(\'scale\',1),\n            \'time_interval_s\':[rows[0][\'time_s\'],rows[-1][\'time_s\']],\n            \'observations\':len(rows),\'track_id\':rows[0][\'track_id\'],\n            \'mapping\':H.tolist(),\'diagnostics\':diagnostics,\'motion_diagnostics\':calculated,\n            \'support_hull_image\':cv2.convexHull(np.array([r[\'quad\'] for r in rows],np.float32).reshape(-1,2)).reshape(-1,2).tolist(),\n            \'method\':\'single_pass_rectangle_calibration\',\'reference_speed_used\':False}\n    if config.get(\'method\')==\'projective_delay\':\n        delay=projective_delay(rows,length)\n        if not delay[\'available\']:return delay\n        result.update(delay)\n        result[\'estimated_kmh\']=result[\'raw_kmh\']*config.get(\'scale\',1)\n    result[\'calibration_length_m\']=length\n    result[\'scale_correction\']=config.get(\'scale\',1)\n    return result\n\n\ndef projective_delay(rows,length_m):\n    """Match front/rear ground positions through a shared 1-D projective motion curve.\n\n    For constant speed, the rear follows the front\'s path after a delay L/v.\n    Fitting this delay estimates speed without reading a reference speed or clip filename.\n    """\n    quads=np.array([r[\'quad\'] for r in rows]);centers=quads.mean(axis=1)\n    direction=centers[-1]-centers[0];direction/=np.linalg.norm(direction)\n    front=(quads[:,1]+quads[:,2])/2;rear=(quads[:,0]+quads[:,3])/2\n    if np.median((front-rear)@direction)<0:front,rear=rear,front\n    front=front@direction;rear=rear@direction\n    times=np.array([r[\'time_s\'] for r in rows]);span=times[-1]-times[0]\n    t=(times-times[0])/span\n    origin=np.mean(np.r_[front,rear]);normalizer=max(np.ptp(np.r_[front,rear]),1)\n    front=(front-origin)/normalizer;rear=(rear-origin)/normalizer\n    linear=np.polyfit(t,front,1)\n\n    def residual(parameters):\n        a,b,c,log_delay=parameters;delay=np.exp(log_delay)\n        def prediction(s):\n            den=1+c*s\n            return (a+b*s)/np.maximum(den,.02),np.maximum(.03-den,0)*10\n        f,fp=prediction(t);r,rp=prediction(t-delay)\n        return np.r_[f-front,r-rear,fp,rp]\n    fits=[]\n    for c in [-.8,-.4,0,.5,1.5]:\n        initial=[linear[1],linear[0],c,math.log(.25/span)]\n        fit=least_squares(residual,initial,bounds=([-5,-20,-.95,math.log(.03/span)],\n                                                 [5,20,4,math.log(4/span)]),loss=\'soft_l1\',f_scale=.02,max_nfev=500)\n        fits.append((np.mean(residual(fit.x)**2),fit))\n    error,fit=min(fits,key=lambda pair:pair[0]);delay_s=float(np.exp(fit.x[3])*span)\n    return {\'available\':True,\'raw_kmh\':length_m/delay_s*3.6,\'method\':\'projective_delay\',\n            \'projective_delay_s\':delay_s,\'projective_fit_rms_pixels\':float(np.sqrt(error)*normalizer),\n            \'projective_parameters\':fit.x.tolist()}\n\n\ndef load_clip(directory):\n    directory=Path(directory)\n    observations=json.loads((directory/\'observations.json\').read_text())\n    report=json.loads((directory/\'geometry_run_report.json\').read_text())\n    # Recompute corner-order voting from observations, including early pilot outputs.\n    points=np.array([r[\'keypoints\'] for r in observations])\n    if len(points):\n        vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n        report[\'ground_indices\']=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n        report[\'ground_order_vote_agreement\']=float(max(vote.mean(),1-vote.mean()))\n    return observations,report\n', 'calibration_candidates.py': '"""Independent per-video calibration candidates; prediction functions accept no speed labels.\n\nThe temporal and rigid-bundle methods are new implementations of projective and\nrigid-body geometry, not reproductions of any authors\' trained keypoint system.\n"""\nimport math\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom benchmark_speed import target_records, projective_delay\n\n\ndef select_rows(observations, report, minimum=100, maximum=400):\n    rows = target_records(observations, report, minimum, maximum)\n    original={(r[\'track_id\'],r[\'frame_index\']):r for r in observations}\n    rows=[{**original[r[\'track_id\'],r[\'frame_index\']],**r} for r in rows]\n    for row in rows:row[\'box_height\']=row[\'baseline_box\'][3]-row[\'baseline_box\'][1]\n    width, height = report[\'width\'], report[\'height\']\n    rows = [r for r in rows if np.all(np.asarray(r[\'quad\'])[:, 0] > width*.02)\n            and np.all(np.asarray(r[\'quad\'])[:, 0] < width*.98)\n            and np.all(np.asarray(r[\'quad\'])[:, 1] < height*.98)]\n    return sorted(rows, key=lambda r:r[\'time_s\'])\n\n\ndef temporal_delay_2d(rows, length_m=4.9):\n    """Fit front/rear ground midpoints to one 2D rational constant-motion curve."""\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    q=np.asarray([r[\'quad\'] for r in rows]);times=np.asarray([r[\'time_s\'] for r in rows])\n    span=float(np.ptp(times))\n    if span<.2:return {\'available\':False,\'reason\':\'short_interval\'}\n    front=(q[:,0]+q[:,3])/2;rear=(q[:,1]+q[:,2])/2\n    direction=np.mean(q[-3:],axis=(0,1))-np.mean(q[:3],axis=(0,1))\n    if np.median((front-rear)@direction)<0:front,rear=rear,front\n    origin=np.mean(np.r_[front,rear],axis=0);scale=max(np.ptp(np.r_[front,rear],axis=0).max(),1)\n    f=(front-origin)/scale;r=(rear-origin)/scale;t=(times-times[0])/span\n    linear=np.array([np.polyfit(t,f[:,axis],1) for axis in [0,1]])\n    def curve(p,s):return (p[:2]+s[:,None]*p[2:4])/np.maximum(1+p[4]*s[:,None],.01)\n    def residual(p):\n        d=np.exp(p[5]);den=np.r_[1+p[4]*t,1+p[4]*(t-d)]\n        return np.r_[(curve(p,t)-f).ravel(),(curve(p,t-d)-r).ravel(),np.maximum(.03-den,0)*10]\n    fits=[]\n    for perspective in [-.7,-.3,0,.7,2]:\n        initial=np.r_[linear[:,1],linear[:,0],perspective,np.log(.25/span)]\n        fit=least_squares(residual,initial,bounds=([-5,-5,-20,-20,-.95,np.log(.03/span)],\n            [5,5,20,20,4,np.log(4/span)]),loss=\'soft_l1\',f_scale=.015,max_nfev=500)\n        error=np.mean(residual(fit.x)**2);fits.append((error,fit))\n    error,fit=min(fits,key=lambda x:x[0]);delay=float(np.exp(fit.x[5])*span)\n    return {\'available\':True,\'estimated_kmh\':length_m/delay*3.6,\'method\':\'temporal_projective_2d\',\n        \'delay_s\':delay,\'time_interval_s\':[float(times[0]),float(times[-1])],\n        \'track_id\':rows[0][\'track_id\'],\'observations\':len(rows),\n        \'reprojection_rms_pixels\':float(np.sqrt(error)*scale),\n        \'parameters\':fit.x.tolist(),\'curve_origin\':origin.tolist(),\'curve_scale\':float(scale),\n        \'fitted_front_image\':(curve(fit.x,t)*scale+origin).tolist(),\n        \'fitted_rear_image\':(curve(fit.x,t-np.exp(fit.x[5]))*scale+origin).tolist(),\n        \'curve_times_s\':times.tolist(),\n        \'support_hull_image\':cv2.convexHull(q.reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'length_prior_m\':length_m,\'reference_label_used\':False}\n\n\ndef corner_vanishing_point(rows):\n    """Robust intersection of tracked longitudinal-corner motion lines."""\n    quads=np.asarray([r[\'quad\'] for r in rows]);lines=[]\n    for index in range(4):\n        vx,vy,x,y=cv2.fitLine(quads[:,index].astype(np.float32),cv2.DIST_HUBER,0,.01,.01).ravel()\n        lines.append([-float(vy),float(vx),float(vy*x-vx*y)])\n    lines=np.array(lines);initial=np.linalg.lstsq(lines[:,:2],-lines[:,2],rcond=None)[0]\n    fit=least_squares(lambda p:lines[:,:2]@p+lines[:,2],initial,loss=\'soft_l1\',f_scale=3)\n    return fit.x,float(np.sqrt(np.mean((lines[:,:2]@fit.x+lines[:,2])**2)))\n\n\ndef shared_vp_delay(rows,length_m=4.9,vanishing_point=None,vp_weight=.02):\n    """Fit both longitudinal sides with a shared motion vanishing point and delay.\n\n    Each side retains its own projective depth. This avoids averaging projected\n    corners before accounting for perspective. An optional flow VP is speed-blind.\n    """\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    q=np.asarray([r[\'quad\'] for r in rows]);times=np.array([r[\'time_s\'] for r in rows])\n    span=float(np.ptp(times));t=(times-times[0])/span\n    fronts=q[:,[0,3]];rears=q[:,[1,2]]\n    direction=q[-1].mean(axis=0)-q[0].mean(axis=0)\n    if np.median(np.sum((fronts-rears)*direction,axis=2))<0:fronts,rears=rears,fronts\n    origin=q.mean(axis=(0,1));scale=max(np.ptp(q.reshape(-1,2),axis=0).max(),1)\n    front=(fronts-origin)/scale;rear=(rears-origin)/scale\n    inferred,uncertainty=corner_vanishing_point(rows)\n    if vanishing_point is None:vanishing_point=inferred\n    vp=(np.asarray(vanishing_point)-origin)/scale\n    # Allow small VP refinement for noisy correspondences while retaining the geometric prior.\n    sigma=max(uncertainty*4,15)/scale\n    def curve(p,s):\n        current_vp=p[7:9];A=p[:4].reshape(2,2);c=p[4:6]\n        return current_vp[None,None]+A[None]/np.maximum(1+s[:,None,None]*c[None,:,None],.02)\n    def residual(p):\n        d=np.exp(p[6]);den=np.r_[(1+t[:,None]*p[4:6]).ravel(),(1+(t[:,None]-d)*p[4:6]).ravel()]\n        return np.r_[(curve(p,t)-front).ravel(),(curve(p,t-d)-rear).ravel(),\n                     np.maximum(.03-den,0)*10,(p[7:9]-vp)/sigma*vp_weight]\n    fits=[]\n    for c in [-.75,-.4,0,.5,2]:\n        initial=np.r_[(front[0]-vp).ravel(),c,c,np.log(.25/span),vp]\n        lower=[-10]*4+[-.95,-.95,np.log(.03/span),-10,-10]\n        upper=[10]*4+[4,4,np.log(4/span),10,10]\n        initial=np.clip(initial,np.array(lower)+1e-6,np.array(upper)-1e-6)\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=.015,max_nfev=500)\n        score=float(np.mean(residual(fit.x)**2));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);delay=float(np.exp(fit.x[6])*span)\n    return {\'available\':True,\'estimated_kmh\':length_m/delay*3.6,\'method\':\'shared_vanishing_point_temporal_calibration\',\n        \'track_id\':rows[0][\'track_id\'],\'observations\':len(rows),\'delay_s\':delay,\'length_prior_m\':length_m,\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'parameters\':fit.x.tolist(),\n        \'vanishing_point_image\':(fit.x[7:9]*scale+origin).tolist(),\n        \'initial_vanishing_point_image\':np.asarray(vanishing_point).tolist(),\n        \'corner_line_rms_pixels\':uncertainty,\'reprojection_rms_pixels\':float(np.sqrt(score)*scale),\n        \'fitted_front_image\':(curve(fit.x,t)*scale+origin).tolist(),\n        \'fitted_rear_image\':(curve(fit.x,t-np.exp(fit.x[6]))*scale+origin).tolist(),\n        \'curve_times_s\':times.tolist(),\n        \'curve_origin\':origin.tolist(),\'curve_scale\':float(scale),\'vanishing_point_weight\':vp_weight,\n        \'reference_label_used\':False}\n\n\ndef rigid_cuboid_bundle(observations,report,length_m=4.9,width_m=1.87,height_m=1.45,distortion=False):\n    """Jointly fit one camera and constant rigid 3D motion to many cuboid observations.\n\n    Camera principal point is assumed at image centre; focal length and rotation\n    are fitted. Object dimensions are priors, not independently measured values.\n    """\n    rows=select_rows(observations,report,100,400)\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    # Bound computation and balance temporal support without reference-speed input.\n    rows=[rows[i] for i in np.unique(np.linspace(0,len(rows)-1,min(36,len(rows))).astype(int))]\n    times=np.array([r[\'time_s\'] for r in rows]);dt=times-times.mean()\n    base=np.array([[-width_m/2,0,0],[-width_m/2,0,length_m],\n                   [width_m/2,0,length_m],[width_m/2,0,0]],float)\n    object_points=np.r_[base,base+[0,-height_m,0]]\n    indices=report[\'ground_indices\']+[i for i in range(8) if i not in report[\'ground_indices\']]\n    points=np.array([r[\'keypoints\'] for r in rows])[:,indices,:2]\n    pixel_scale=np.maximum(np.array([r[\'box_height\'] for r in rows]),100)\n    center=np.median(points[:,:4].mean(axis=1),axis=0);cx=report[\'width\']/2;cy=report[\'height\']/2\n    def camera(p):\n        R=cv2.Rodrigues(p[:3])[0];T=np.r_[p[3:5],np.exp(p[5])]\n        v=np.exp(p[6]);f=np.exp(p[7]);k=p[8] if distortion else 0\n        XYZ=object_points[None]@R.T+T-dt[:,None,None]*v*R[:,2]\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)\n        xy*=1+k*np.sum(xy**2,axis=2,keepdims=True)\n        image=xy*f+[cx,cy]\n        return image,XYZ,R,T,f\n    def residual(p):\n        image,XYZ,*_=camera(p)\n        return np.r_[((image-points)/pixel_scale[:,None,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n    lower=[-.8,-1.5,-.4,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.8,1.5,.4,100,10,np.log(1000),np.log(70),np.log(4000)]\n    if distortion:lower+=[-.5];upper+=[.5]\n    fits=[]\n    for focal,yaw in [(600,0),(1000,0),(1600,0),(1000,.35),(1000,-.35)]:\n        depth=20;initial=[0,yaw,0,(center[0]-cx)/focal*depth,(center[1]-cy)/focal*depth,\n                         np.log(depth),np.log(12),np.log(focal)]\n        if distortion:initial+=[0]\n        initial=np.clip(initial,np.array(lower)+1e-6,np.array(upper)-1e-6)\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=3,max_nfev=300)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,100**2)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);image,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]])\n    H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[6])*3.6),\n        \'method\':\'rigid_cuboid_bundle_distortion\' if distortion else \'rigid_cuboid_bundle\',\n        \'parameters\':fit.x.tolist(),\'focal_length_pixels\':float(f),\'radial_k1\':float(fit.x[8]) if distortion else 0,\n        \'mapping\':H.tolist(),\'support_hull_image\':cv2.convexHull(points[:,:4].reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'track_id\':rows[0][\'track_id\'],\n        \'observations\':len(rows),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'dimensions_m\':[length_m,width_m,height_m],\'reference_label_used\':False}\n\n\ndef silhouette_bundle(boxes,times,width,height,shape=\'sedan\',length_m=4.9,width_m=1.87,height_m=1.45):\n    """Fit a rigid constant-motion vehicle envelope directly to detector boxes.\n\n    This avoids inferred hidden cuboid corners. The generic envelope is a size\n    prior, not an identified CAD model of the Mercedes.\n    """\n    boxes=np.asarray(boxes,float);times=np.asarray(times,float)\n    keep=(boxes[:,3]-boxes[:,1]>=70)&(boxes[:,3]-boxes[:,1]<=450)&(boxes[:,0]>width*.02)&(boxes[:,2]<width*.98)\n    boxes=boxes[keep];times=times[keep]\n    if len(boxes)<8:return {\'available\':False,\'reason\':\'insufficient_unclipped_boxes\'}\n    selected=np.unique(np.linspace(0,len(boxes)-1,min(60,len(boxes))).astype(int))\n    boxes=boxes[selected];times=times[selected];dt=times-times.mean()\n    if shape==\'cuboid\':\n        vertices=np.array([[x,y,z] for x in [-width_m/2,width_m/2]\n                           for y in [0,-height_m] for z in [0,length_m]])\n    else:\n        # Front/rear bumper, bonnet, windscreen, roof and rear deck sections.\n        sections=[(0,.85,1),(.22,1.0,1),(1.25,1.0,.98),(1.8,height_m,.77),\n                  (3.2,height_m,.77),(3.85,1.0,.96),(length_m,.83,1)]\n        vertices=np.array([[sign*width_m/2*w,-h,z] for z,h,w in sections for sign in [-1,1]]+\n                          [[sign*width_m/2,0,z] for z in [.85,length_m-.85] for sign in [-1,1]])\n    cx=width/2;cy=height/2;center=np.median((boxes[:,:2]+boxes[:,2:])/2,axis=0)\n    pixel_scale=np.maximum(boxes[:,3]-boxes[:,1],70)\n    def camera(p):\n        R=cv2.Rodrigues(p[:3])[0];T=np.r_[p[3:5],np.exp(p[5])];v=np.exp(p[6]);f=np.exp(p[7])\n        XYZ=vertices[None]@R.T+T-dt[:,None,None]*v*R[:,2]\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)*f+[cx,cy]\n        predicted=np.c_[xy.min(axis=1),xy.max(axis=1)]\n        return predicted,XYZ,R,T,f\n    def residual(p):\n        predicted,XYZ,*_=camera(p)\n        return np.r_[((predicted-boxes)/pixel_scale[:,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n    lower=[-.6,-1.5,-.2,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.6,1.5,.2,100,10,np.log(1000),np.log(70),np.log(4000)]\n    fits=[]\n    for focal,yaw in [(600,0),(1000,0),(1600,0),(1000,.4),(1000,-.4)]:\n        initial=[0,yaw,0,(center[0]-cx)/focal*20,(center[1]-cy)/focal*20,\n                 np.log(20),np.log(12),np.log(focal)]\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=2,max_nfev=500)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,10000)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);predicted,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]])\n    H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    anchors=np.c_[(boxes[:,0]+boxes[:,2])/2,boxes[:,3]]\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[6])*3.6),\n        \'method\':\'silhouette_bundle_\'+shape,\'parameters\':fit.x.tolist(),\'mapping\':H.tolist(),\n        \'focal_length_pixels\':float(f),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'time_interval_s\':[float(times[0]),float(times[-1])],\'observations\':len(boxes),\n        \'fitted_boxes\':predicted.tolist(),\'observed_boxes\':boxes.tolist(),\'times_s\':times.tolist(),\n        \'support_hull_image\':cv2.convexHull(anchors.astype(np.float32)).reshape(-1,2).tolist(),\n        \'dimensions_m\':[length_m,width_m,height_m],\'reference_label_used\':False}\n\n\ndef vp_rigid_bundle(observations,report,vanishing_point,length_m=4.9,width_m=1.87,height_m=1.45,fit_length=False):\n    """Rigid camera fit whose translation axis is fixed by visual optical flow."""\n    rows=select_rows(observations,report)\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_geometry\'}\n    rows=[rows[i] for i in np.unique(np.linspace(0,len(rows)-1,min(36,len(rows))).astype(int))]\n    times=np.array([r[\'time_s\'] for r in rows]);dt=times-times.mean()\n    base=np.array([[-width_m/2,0,0],[-width_m/2,0,length_m],[width_m/2,0,length_m],[width_m/2,0,0]])\n    vertices=np.r_[base,base+[0,-height_m,0]]\n    indices=report[\'ground_indices\']+[i for i in range(8) if i not in report[\'ground_indices\']]\n    observed=np.array([r[\'keypoints\'] for r in rows])[:,indices,:2]\n    pixel_scale=np.array([r[\'box_height\'] for r in rows]);cx=report[\'width\']/2;cy=report[\'height\']/2\n    vp=np.asarray(vanishing_point);center=np.median(observed[:,:4].mean(axis=1),axis=0)\n    def camera(p):\n        roll,tx,ty,logz,logv,logf=p[:6];f=np.exp(logf)\n        z=np.r_[(vp-[cx,cy])/f,1];z/=np.linalg.norm(z)\n        x=np.cross([0,1,0],z);x/=np.linalg.norm(x);y=np.cross(z,x)\n        R=np.c_[x*np.cos(roll)+y*np.sin(roll),-x*np.sin(roll)+y*np.cos(roll),z]\n        T=np.array([tx,ty,np.exp(logz)]);v=np.exp(logv)\n        current=vertices.copy()\n        if fit_length:current[:,2]*=np.exp(p[6])/length_m\n        XYZ=current[None]@R.T+T-dt[:,None,None]*v*z\n        xy=XYZ[:,:,:2]/np.maximum(XYZ[:,:,2:],.1)*f+[cx,cy]\n        return xy,XYZ,R,T,f\n    def residual(p):\n        xy,XYZ,*_=camera(p)\n        values=np.r_[((xy-observed)/pixel_scale[:,None,None]*200).ravel(),np.maximum(1-XYZ[:,:,2],0).ravel()*20]\n        return np.r_[values,(p[6]-np.log(length_m))] if fit_length else values\n    lower=[-.4,-100,-10,np.log(3),np.log(.5),np.log(250)]\n    upper=[.4,100,10,np.log(1000),np.log(70),np.log(4000)]\n    if fit_length:lower+=[np.log(2.5)];upper+=[np.log(7.5)]\n    fits=[]\n    for f in [600,1000,1600]:\n        initial=[0,(center[0]-cx)/f*20,(center[1]-cy)/f*20,np.log(20),np.log(12),np.log(f)]\n        if fit_length:initial+=[np.log(length_m)]\n        fit=least_squares(residual,initial,bounds=(lower,upper),loss=\'soft_l1\',f_scale=3,max_nfev=400)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,10000)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);xy,XYZ,R,T,f=camera(fit.x)\n    K=np.array([[f,0,cx],[0,f,cy],[0,0,1]]);H=np.linalg.inv(K@np.c_[R[:,0],R[:,2],T])\n    return {\'available\':bool(np.min(XYZ[:,:,2])>0),\'estimated_kmh\':float(np.exp(fit.x[4])*3.6),\n        \'method\':\'optical_flow_vp_rigid_bundle\',\'mapping\':H.tolist(),\'parameters\':fit.x.tolist(),\n        \'focal_length_pixels\':float(f),\'vanishing_point_image\':vp.tolist(),\n        \'track_id\':rows[0][\'track_id\'],\'time_interval_s\':[float(times[0]),float(times[-1])],\n        \'observations\':len(rows),\'weighted_reprojection_rms\':float(np.sqrt(score)),\n        \'support_hull_image\':cv2.convexHull(observed[:,:4].reshape(-1,2).astype(np.float32)).reshape(-1,2).tolist(),\n        \'reference_label_used\':False,\'dimensions_m\':[float(np.exp(fit.x[6])) if fit_length else length_m,width_m,height_m],\n        \'effective_cuboid_length_fitted\':fit_length}\n', 'flow_calibration.py': '"""Forward/backward checked optical flow and robust contact-anchor transport.\n\nInspired by dynamic vehicle-template flow, with the metric ground map supplied\nby a separate calibration method. This is not the 36-keypoint paper\'s model.\n"""\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom benchmark_speed import target_records\nfrom calibrate_speed import project\n\n\ndef extract_flow(video,observations,report,tracks):\n    rows=target_records(observations,report)\n    if not rows:return {\'available\':False,\'measurements\':[]}\n    tid=rows[0][\'track_id\'];by_frame={r[\'frame_index\']:r for r in tracks if r[\'track_id\']==tid}\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened()\n    ok,previous=cap.read();index=0;measurements=[]\n    try:\n        while ok:\n            ok,current=cap.read()\n            if not ok:break\n            a=by_frame.get(index);b=by_frame.get(index+1)\n            if a is not None and b is not None:\n                box=np.array(a[\'box\']);next_box=np.array(b[\'box\']);height=box[3]-box[1]\n                if 80<=height<=450 and min(box[0],report[\'width\']-box[2])>report[\'width\']*.02:\n                    union=np.r_[np.minimum(box[:2],next_box[:2])-20,np.maximum(box[2:],next_box[2:])+20]\n                    x0,y0,x1,y1=np.rint(union).astype(int)\n                    x0=max(x0,0);y0=max(y0,0);x1=min(x1,report[\'width\']);y1=min(y1,report[\'height\'])\n                    g0=cv2.cvtColor(previous[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                    g1=cv2.cvtColor(current[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                    mask=np.zeros_like(g0);offset=np.array([x0,y0])\n                    p0=(box[:2]+(box[2:]-box[:2])*.12-offset).astype(int)\n                    p1=(box[2:]-(box[2:]-box[:2])*.12-offset).astype(int)\n                    cv2.rectangle(mask,tuple(p0),tuple(p1),255,-1)\n                    features=cv2.goodFeaturesToTrack(g0,120,.01,4,mask=mask)\n                    if features is not None and len(features)>=8:\n                        forward,status,_=cv2.calcOpticalFlowPyrLK(g0,g1,features,None,winSize=(31,31),maxLevel=3)\n                        backward,bs,_=cv2.calcOpticalFlowPyrLK(g1,g0,forward,None,winSize=(31,31),maxLevel=3)\n                        old=features[:,0];new=forward[:,0]\n                        keep=(status[:,0]>0)&(bs[:,0]>0)&(np.linalg.norm(backward[:,0]-old,axis=1)<1)\n                        keep&=(np.linalg.norm(new-old,axis=1)>.25)\n                        if keep.sum()>=6:\n                            affine,inliers=cv2.estimateAffinePartial2D(old[keep],new[keep],method=cv2.RANSAC,ransacReprojThreshold=1.5,maxIters=1000)\n                            if affine is not None and inliers.sum()>=6:\n                                anchor=np.array([(box[0]+box[2])/2,box[3]])\n                                transported=affine@np.r_[anchor-offset,1]+offset\n                                selected=inliers[:,0].astype(bool)\n                                error=np.linalg.norm(old[keep][selected]@affine[:,:2].T+affine[:,2]-new[keep][selected],axis=1)\n                                measurements.append({\'frame_index\':index,\'time_s\':a[\'time_s\'],\n                                    \'dt_s\':b[\'time_s\']-a[\'time_s\'],\'anchor\':anchor.tolist(),\n                                    \'transported_anchor\':transported.tolist(),\'affine\':affine.tolist(),\n                                    \'crop_origin\':offset.tolist(),\'inliers\':int(inliers.sum()),\n                                    \'flow_rms_pixels\':float(np.sqrt(np.mean(error**2))),\n                                    \'box_height\':float(height)})\n            previous=current;index+=1\n    finally:cap.release()\n    return {\'available\':len(measurements)>=8,\'track_id\':tid,\'measurements\':measurements,\n            \'reference_label_used\':False,\'frames_decoded\':index+1}\n\n\ndef estimate_flow(features,mapping,trim_fraction=.1):\n    rows=features.get(\'measurements\',[])\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_checked_flow\'}\n    times=np.array([r[\'time_s\'] for r in rows]);lo,hi=np.quantile(times,[trim_fraction,1-trim_fraction])\n    rows=[r for r in rows if lo<=r[\'time_s\']<=hi]\n    points=np.array([r[\'anchor\'] for r in rows]);next_points=np.array([r[\'transported_anchor\'] for r in rows])\n    dt=np.array([r[\'dt_s\'] for r in rows])\n    velocities=(project(next_points,np.array(mapping))-project(points,np.array(mapping)))/dt[:,None]\n    speeds=np.linalg.norm(velocities,axis=1)*3.6\n    finite=np.isfinite(speeds)&(speeds>0)&(speeds<300)\n    if finite.sum()<8:return {\'available\':False,\'reason\':\'invalid_mapped_flow\'}\n    median=np.median(speeds[finite]);mad=np.median(np.abs(speeds[finite]-median))\n    keep=finite&(np.abs(speeds-median)<=max(4*1.4826*mad,2))\n    return {\'available\':True,\'estimated_kmh\':float(np.median(speeds[keep])),\n        \'method\':\'learned_map_and_optical_flow\',\'mapping\':np.asarray(mapping).tolist(),\n        \'track_id\':features[\'track_id\'],\'observations\':int(keep.sum()),\n        \'time_interval_s\':[float(times.min()),float(times.max())],\n        \'support_hull_image\':cv2.convexHull(points.astype(np.float32)).reshape(-1,2).tolist(),\n        \'per_frame_kmh\':speeds.tolist(),\'per_frame_keep\':keep.tolist(),\n        \'per_frame_times_s\':[r[\'time_s\'] for r in rows],\n        \'speed_mad_kmh\':float(mad),\'time_trim_fraction\':trim_fraction,\'reference_label_used\':False}\n', 'learned_calibration.py': '"""Adapter for Revaud/Humenberger ICCV 2021\'s released calibration network.\n\nUpstream implementation is downloaded at a pinned revision into ignored runtime\nstorage and retains its CC BY-NC-SA 3.0 terms. No upstream weights are committed.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport urllib.request\n\nimport cv2\nimport numpy as np\nfrom benchmark_speed import load_clip, target_records\nfrom calibrate_speed import project\n\nUPSTREAM=\'https://raw.githubusercontent.com/naver/cctv/\'\nREVISION=\'80b8bc4964eddfda32c40e6d1c8fe8b62f777f55\'\nWEIGHT_SHA=\'bcb5af76b769788ec18b663e0e1a879ab2d3dce30eb2d0f3162e9730e5204fbf\'\nFILES=[\'archs/__init__.py\',\'archs/transformer.py\',\'tools/geometry.py\',\'tools/speed.py\',\n       \'tools/common.py\',\'tools/tracks.py\',\'LICENSE\',\'NOTICE\',\'models/trained_transformer.pt\']\n\n\ndef prepare_runtime(root):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    for name in FILES:\n        path=root/name;path.parent.mkdir(parents=True,exist_ok=True)\n        if not path.exists():urllib.request.urlretrieve(UPSTREAM+REVISION+\'/\'+name,path)\n    weights=root/\'models/trained_transformer.pt\'\n    assert hashlib.sha256(weights.read_bytes()).hexdigest()==WEIGHT_SHA\n    if str(root.resolve()) not in sys.path:sys.path.insert(0,str(root.resolve()))\n    import torch\n    from archs.transformer import Transformer\n    data=torch.load(weights,map_location=\'cpu\',weights_only=True)\n    assert data[\'arch\']==\'Transformer(nlayers=8, nhead=4, nhid=256)\'\n    net=Transformer(embdim=64,outdim=8,nlayers=8,nhead=4,nhid=256).eval()\n    net.load_state_dict(data[\'weights\'])\n    return net\n\n\ndef iou(box,boxes):\n    lo=np.maximum(box[:2],boxes[:,:2]);hi=np.minimum(box[2:],boxes[:,2:])\n    area=np.maximum(hi-lo,0).prod(axis=1)\n    return area/(np.prod(box[2:]-box[:2])+np.prod(boxes[:,2:]-boxes[:,:2],axis=1)-area+1e-9)\n\n\ndef extract_mask_features(video,observations,report,detector,device,max_samples=24):\n    """Get segmentation moments and motion direction without any reference speed."""\n    import torch\n    rows=target_records(observations,report,70,450)\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_track\'},None\n    original={(r[\'track_id\'],r[\'frame_index\']):r for r in observations}\n    selected=[rows[i] for i in np.unique(np.linspace(0,len(rows)-1,min(max_samples,len(rows))).astype(int))]\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened();features=[];preview=None\n    W,H=report[\'width\'],report[\'height\'];S=max(W,H)\n    try:\n        for row in selected:\n            cap.set(cv2.CAP_PROP_POS_FRAMES,row[\'frame_index\']);ok,frame=cap.read();assert ok\n            tensor=torch.from_numpy(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)).permute(2,0,1).float().to(device)/255\n            with torch.inference_mode():result=detector([tensor])[0]\n            accepted=(result[\'labels\']==3)&(result[\'scores\']>=.5)\n            boxes=result[\'boxes\'][accepted].cpu().numpy()\n            if not len(boxes):continue\n            baseline=np.array(original[row[\'track_id\'],row[\'frame_index\']][\'baseline_box\'])\n            scores=iou(baseline,boxes);index=int(np.argmax(scores))\n            if scores[index]<.3:continue\n            box=boxes[index];mask=result[\'masks\'][accepted][index,0].cpu().numpy()\n            x0,y0,x1,y1=np.rint(box).astype(int);x0=max(0,x0);y0=max(0,y0);x1=min(W,x1);y1=min(H,y1)\n            if min(x1-x0,y1-y0)<10:continue\n            small=cv2.resize(mask[y0:y1,x0:x1],(28,28),interpolation=cv2.INTER_AREA)\n            xy=np.mgrid[0:28,0:28].reshape(2,-1)[::-1].T\n            xy=(xy+.5)/28*(box[2:]-box[:2])/S+box[:2]/S\n            weight=small.ravel();weight/=max(weight.sum(),1e-9)\n            center=(weight[:,None]*xy).sum(axis=0);diff=xy-center\n            covariance=(diff.T*weight)@diff\n            features.append({\'frame_index\':row[\'frame_index\'],\'time_s\':row[\'time_s\'],\'box\':box.tolist(),\n                \'center_normalized\':center.tolist(),\'covariance_normalized\':covariance.ravel().tolist(),\n                \'association_iou\':float(scores[index])})\n            if len(features)==len(selected)//2:\n                preview=frame.copy();preview[mask>.5]=(preview[mask>.5]*.5+np.array([0,127,0])).astype(np.uint8)\n    finally:cap.release()\n    return {\'available\':len(features)>=8,\'track_id\':rows[0][\'track_id\'],\'features\':features,\n        \'width\':W,\'height\':H,\'fps\':report[\'fps\'],\'model\':\'torchvision MaskRCNN ResNet50 FPN COCO V1\',\n        \'segmentation_min_size\':640,\'reference_label_used\':False},preview\n\n\ndef calibrate_from_features(features,net,ensemble=1,seed=17):\n    import torch\n    from tools.speed import ransac_homography_from_jacobians\n    rows=features.get(\'features\',[])\n    if len(rows)<8:return {\'available\':False,\'reason\':\'insufficient_segmented_observations\'}\n    centers=np.array([r[\'center_normalized\'] for r in rows]);cov=np.array([r[\'covariance_normalized\'] for r in rows])\n    times=np.array([r[\'time_s\'] for r in rows]);motion=np.gradient(centers,times,axis=0)\n    motion/=np.maximum(np.linalg.norm(motion,axis=1,keepdims=True),1e-9)\n    motion*=np.sign(motion[:,1:2])\n    embeddings=np.c_[centers,1001*cov,motion].astype(np.float32)\n    rng=np.random.default_rng(seed);all_predictions=[]\n    for repeat in range(ensemble):\n        indices=(np.unique(np.linspace(0,len(rows)-1,min(10,len(rows))).astype(int)) if ensemble==1\n                 else rng.choice(len(rows),size=min(10,len(rows)),replace=False))\n        with torch.inference_mode():prediction=net(torch.from_numpy(embeddings[indices][None]))[0]\n        all_predictions.append(prediction)\n    np.random.seed(seed)\n    H=ransac_homography_from_jacobians(torch.cat(all_predictions),niter=1024)\n    H=H@np.diag([1/max(features[\'width\'],features[\'height\'])]*2+[1])\n    boxes=np.array([r[\'box\'] for r in rows]);anchors=np.c_[(boxes[:,0]+boxes[:,2])/2,boxes[:,3]]\n    world=project(anchors,H);t=times-times.mean()\n    velocity=np.array([np.polyfit(t,world[:,axis],1)[0] for axis in [0,1]])\n    predicted=world.mean(axis=0)+t[:,None]*velocity\n    return {\'available\':bool(np.isfinite(world).all()),\'estimated_kmh\':float(np.linalg.norm(velocity)*3.6),\n        \'method\':\'learned_mask_calibration\' if ensemble==1 else \'learned_mask_calibration_ensemble\',\n        \'mapping\':H.tolist(),\'time_interval_s\':[float(times[0]),float(times[-1])],\n        \'support_hull_image\':cv2.convexHull(anchors.astype(np.float32)).reshape(-1,2).tolist(),\n        \'track_id\':features[\'track_id\'],\'observations\':len(rows),\'ensemble_samples\':ensemble,\n        \'trajectory_rms_m\':float(np.sqrt(np.mean((world-predicted)**2))),\n        \'upstream_revision\':REVISION,\'upstream_model_sha256\':WEIGHT_SHA,\'reference_label_used\':False}\n\n\ndef run(args):\n    import torch\n    from torchvision.models.detection import maskrcnn_resnet50_fpn,MaskRCNN_ResNet50_FPN_Weights\n    torch.set_num_threads(4)\n    net=prepare_runtime(args.runtime)\n    detector=maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,min_size=640,max_size=1280).eval().to(args.device)\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    folders=sorted(Path(args.inference_dir).iterdir())\n    if args.pilots:folders=[p for p in folders if p.name in {f\'MercedesAMG550_{s}\' for s in [30,60,100]}]\n    for folder in folders:\n        if not (folder/\'observations.json\').exists():continue\n        target=out/folder.name;target.mkdir(exist_ok=True)\n        obs,report=load_clip(folder)\n        if (target/\'mask_features.json\').exists():features=json.loads((target/\'mask_features.json\').read_text())\n        else:\n            features,preview=extract_mask_features(Path(args.video_dir)/(folder.name+\'.MP4\'),obs,report,detector,args.device)\n            (target/\'mask_features.json\').write_text(json.dumps(features,indent=2),encoding=\'utf-8\')\n            if preview is not None:cv2.imwrite(str(target/\'mask_example.jpg\'),cv2.resize(preview,(1280,720)))\n        results={f\'learned_{n}\':calibrate_from_features(features,net,n) for n in [1,16]}\n        (target/\'learned_predictions.json\').write_text(json.dumps(results,indent=2),encoding=\'utf-8\')\n        print(folder.name,\'mask observations\',len(features.get(\'features\',[])),\n            \'estimates\',{k:round(v[\'estimated_kmh\'],2) if v.get(\'estimated_kmh\') is not None else None for k,v in results.items()},flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in [\'inference-dir\',\'video-dir\',\'output\',\'runtime\']:parser.add_argument(\'--\'+name,required=True)\n    parser.add_argument(\'--device\',default=\'cuda\');parser.add_argument(\'--pilots\',action=\'store_true\')\n    run(parser.parse_args())\n', 'research_inference.py': '"""Run multiple speed-blind per-video calibration techniques and archive each version."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\nimport cv2\nimport numpy as np\n\nfrom benchmark_speed import load_clip, target_records, projective_delay\nfrom calibration_candidates import select_rows,temporal_delay_2d,rigid_cuboid_bundle,silhouette_bundle\nfrom flow_calibration import extract_flow,estimate_flow\nfrom learned_calibration import prepare_runtime,extract_mask_features,calibrate_from_features\n\n\ndef run(args):\n    import torch,torchvision\n    from torchvision.models.detection import maskrcnn_resnet50_fpn,MaskRCNN_ResNet50_FPN_Weights\n    torch.set_num_threads(4);cv2.setNumThreads(4)\n    net=prepare_runtime(args.runtime)\n    detector=maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,min_size=640,max_size=1280).eval().to(args.device)\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    for folder in sorted(Path(args.inference_dir).iterdir()):\n        if not (folder/\'observations.json\').exists():continue\n        target=out/folder.name;target.mkdir(exist_ok=True);start=time.monotonic()\n        obs,report=load_clip(folder);tracks=json.loads((folder/\'tracks.json\').read_text())\n        video=Path(args.video_dir)/(folder.name+\'.MP4\')\n        def cached(name,call):\n            path=target/(name+\'.json\')\n            if path.exists():return json.loads(path.read_text())\n            value=call();path.write_text(json.dumps(value,indent=2),encoding=\'utf-8\');return value\n        def masks():\n            features,preview=extract_mask_features(video,obs,report,detector,args.device)\n            if preview is not None:cv2.imwrite(str(target/\'mask_example.jpg\'),cv2.resize(preview,(1280,720)))\n            return features\n        mask=cached(\'mask_features\',masks)\n        flow=cached(\'flow_features\',lambda:extract_flow(video,obs,report,tracks))\n        rows=select_rows(obs,report,80,300)\n        delay=projective_delay(rows,4.9) if len(rows)>=8 else {\'available\':False}\n        if delay.get(\'available\'):\n            delay.update(estimated_kmh=delay[\'raw_kmh\'],track_id=rows[0][\'track_id\'],\n                time_interval_s=[rows[0][\'time_s\'],rows[-1][\'time_s\']])\n        predictions={\'v2_delay_1d\':delay,\n            \'v2b_delay_2d\':cached(\'delay_2d\',lambda:temporal_delay_2d(select_rows(obs,report))),\n            \'v3_rigid_bundle\':cached(\'rigid_bundle\',lambda:rigid_cuboid_bundle(obs,report)),\n            \'v3b_rigid_distortion\':cached(\'rigid_distortion\',lambda:rigid_cuboid_bundle(obs,report,distortion=True)),\n            \'v4_learned_mask\':cached(\'learned_mask\',lambda:calibrate_from_features(mask,net,1)),\n            \'v4b_learned_ensemble\':cached(\'learned_ensemble\',lambda:calibrate_from_features(mask,net,16))}\n        target_rows=target_records(obs,report)\n        tid=target_rows[0][\'track_id\'] if target_rows else None\n        tracking=[r for r in tracks if r[\'track_id\']==tid]\n        for shape in [\'cuboid\',\'sedan\']:\n            result=cached(\'silhouette_\'+shape,lambda: silhouette_bundle([r[\'box\'] for r in tracking],\n                [r[\'time_s\'] for r in tracking],report[\'width\'],report[\'height\'],shape))\n            result[\'track_id\']=tid;predictions[\'v6_silhouette_\'+shape]=result\n        for version,key in [(\'v5_learned_flow\',\'v4_learned_mask\'),(\'v5b_ensemble_flow\',\'v4b_learned_ensemble\')]:\n            learned=predictions[key]\n            predictions[version]=estimate_flow(flow,learned[\'mapping\']) if learned.get(\'available\') else {\'available\':False,\'reason\':\'calibration_unavailable\'}\n        (target/\'predictions.json\').write_text(json.dumps(predictions,indent=2),encoding=\'utf-8\')\n        run={\'video\':video.name,\'reference_speeds_used\':False,\'runtime_s\':time.monotonic()-start,\n            \'torch\':torch.__version__,\'torchvision\':torchvision.__version__,\'opencv\':cv2.__version__,\n            \'gpu\':torch.cuda.get_device_name() if torch.cuda.is_available() else None}\n        (target/\'run_report.json\').write_text(json.dumps(run,indent=2),encoding=\'utf-8\')\n        print(folder.name,{k:round(v[\'estimated_kmh\'],1) if v.get(\'estimated_kmh\') is not None else None for k,v in predictions.items()},flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in [\'inference-dir\',\'video-dir\',\'output\',\'runtime\']:parser.add_argument(\'--\'+name,required=True)\n    parser.add_argument(\'--device\',default=\'cuda\')\n    run(parser.parse_args())\n', 'refine_flow.py': '"""Adaptive optical-flow intervals and speed-blind constant-motion map refinement."""\nfrom collections import deque\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom benchmark_speed import target_records\nfrom calibrate_speed import project\nfrom flow_calibration import estimate_flow\n\n\ndef extract_adaptive_flow(video,observations,report,tracks,max_gap=4,min_displacement=4):\n    rows=target_records(observations,report)\n    if not rows:return {\'available\':False,\'measurements\':[]}\n    tid=rows[0][\'track_id\'];by_frame={r[\'frame_index\']:r for r in tracks if r[\'track_id\']==tid}\n    cap=cv2.VideoCapture(str(video));assert cap.isOpened();buffer=deque(maxlen=max_gap+1)\n    measurements=[];index=0\n    def anchor(box):return np.array([(box[0]+box[2])/2,box[3]])\n    try:\n        while True:\n            ok,frame=cap.read()\n            if not ok:break\n            buffer.append((index,frame))\n            b=by_frame.get(index)\n            if b is not None:\n                for gap in range(1,min(max_gap,index)+1):\n                    a=by_frame.get(index-gap)\n                    if a is None:continue\n                    if np.linalg.norm(anchor(a[\'box\'])-anchor(b[\'box\']))>=min_displacement or gap==max_gap:\n                        break\n                else:a=None\n                if a is not None:\n                    old_index=index-gap;previous=next(im for number,im in buffer if number==old_index)\n                    box=np.array(a[\'box\']);next_box=np.array(b[\'box\']);height=box[3]-box[1]\n                    if 80<=height<=450 and min(box[0],report[\'width\']-box[2])>report[\'width\']*.02:\n                        union=np.r_[np.minimum(box[:2],next_box[:2])-20,np.maximum(box[2:],next_box[2:])+20]\n                        x0,y0,x1,y1=np.rint(union).astype(int);x0=max(x0,0);y0=max(y0,0)\n                        x1=min(x1,report[\'width\']);y1=min(y1,report[\'height\']);offset=np.array([x0,y0])\n                        g0=cv2.cvtColor(previous[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                        g1=cv2.cvtColor(frame[y0:y1,x0:x1],cv2.COLOR_BGR2GRAY)\n                        mask=np.zeros_like(g0)\n                        cv2.rectangle(mask,tuple((box[:2]+(box[2:]-box[:2])*.12-offset).astype(int)),\n                            tuple((box[2:]-(box[2:]-box[:2])*.12-offset).astype(int)),255,-1)\n                        features=cv2.goodFeaturesToTrack(g0,120,.01,4,mask=mask)\n                        if features is not None and len(features)>=8:\n                            forward,status,_=cv2.calcOpticalFlowPyrLK(g0,g1,features,None,winSize=(31,31),maxLevel=3)\n                            backward,bs,_=cv2.calcOpticalFlowPyrLK(g1,g0,forward,None,winSize=(31,31),maxLevel=3)\n                            old=features[:,0];new=forward[:,0]\n                            keep=(status[:,0]>0)&(bs[:,0]>0)&(np.linalg.norm(backward[:,0]-old,axis=1)<1)\n                            keep&=(np.linalg.norm(new-old,axis=1)>.25)\n                            if keep.sum()>=6:\n                                affine,inliers=cv2.estimateAffinePartial2D(old[keep],new[keep],method=cv2.RANSAC,ransacReprojThreshold=1.5,maxIters=1000)\n                                if affine is not None and inliers.sum()>=6:\n                                    point=anchor(box);transported=affine@np.r_[point-offset,1]+offset\n                                    valid=inliers[:,0].astype(bool)\n                                    error=np.linalg.norm(old[keep][valid]@affine[:,:2].T+affine[:,2]-new[keep][valid],axis=1)\n                                    measurements.append({\'frame_index\':old_index,\'next_frame_index\':index,\n                                        \'frame_gap\':gap,\'time_s\':a[\'time_s\'],\'dt_s\':b[\'time_s\']-a[\'time_s\'],\n                                        \'anchor\':point.tolist(),\'transported_anchor\':transported.tolist(),\n                                        \'affine\':affine.tolist(),\'crop_origin\':offset.tolist(),\n                                        \'matched_points_image\':(old[keep][valid]+offset).tolist(),\n                                        \'matched_next_points_image\':(new[keep][valid]+offset).tolist(),\n                                        \'inliers\':int(inliers.sum()),\'flow_rms_pixels\':float(np.sqrt(np.mean(error**2))),\n                                        \'box_height\':float(height)})\n            index+=1\n    finally:cap.release()\n    return {\'available\':len(measurements)>=8,\'track_id\':tid,\'measurements\':measurements,\n            \'frames_decoded\':index,\'maximum_frame_gap\':max_gap,\'minimum_target_pixel_displacement\':min_displacement,\n            \'reference_label_used\':False}\n\n\ndef flow_vanishing_point(features):\n    points=[];weights=[]\n    for row in features.get(\'measurements\',[]):\n        A=np.asarray(row[\'affine\']);C=np.eye(2)-A[:,:2]\n        if abs(np.linalg.det(C))<1e-6:continue\n        center=np.linalg.solve(C,A[:,2]+C@np.asarray(row[\'crop_origin\']))\n        if not np.isfinite(center).all() or abs(center).max()>10000:continue\n        points.append(center);weights.append(row[\'inliers\'])\n    if len(points)<8:return {\'available\':False,\'reason\':\'insufficient_expansion_centres\'}\n    points=np.asarray(points);initial=np.median(points,axis=0)\n    mad=np.median(np.abs(points-initial),axis=0)\n    keep=np.all(np.abs(points-initial)<=np.maximum(4*1.4826*mad,15),axis=1)\n    point=np.median(points[keep],axis=0)\n    return {\'available\':True,\'point\':point.tolist(),\'observations\':int(keep.sum()),\n            \'median_absolute_deviation_pixels\':np.median(np.abs(points[keep]-point),axis=0).tolist(),\n            \'reference_label_used\':False}\n\n\ndef refine_constant_motion(features,mapping):\n    """Constrain mapped flow to one velocity without knowing its magnitude.\n\n    Retain the learned map\'s Jacobian at the centre of observed support to fix\n    metric scale. This assumption is specific to these constant-speed passes.\n    """\n    rows=features.get(\'measurements\',[])\n    if len(rows)<12:return {\'available\':False,\'reason\':\'insufficient_checked_flow\'}\n    times=np.array([r[\'time_s\'] for r in rows]);lo,hi=np.quantile(times,[.1,.9])\n    rows=[r for r in rows if lo<=r[\'time_s\']<=hi]\n    points=np.array([r[\'anchor\'] for r in rows]);next_points=np.array([r[\'transported_anchor\'] for r in rows])\n    dt=np.array([r[\'dt_s\'] for r in rows]);H=np.asarray(mapping)\n    world=project(points,H);after=project(next_points,H)\n    origin=np.median(world,axis=0);scale=max(np.ptp(world,axis=0).max(),1)\n    normalizer=np.array([[1/scale,0,-origin[0]/scale],[0,1/scale,-origin[1]/scale],[0,0,1]])\n    w=project(world,normalizer);a=project(after,normalizer)\n    velocity=(a-w)/dt[:,None];median=np.median(velocity,axis=0)\n    speed=max(np.linalg.norm(median),1e-5)\n    def warp(p):return np.array([[1,0,0],[0,1,0],[p[0],p[1],1]])\n    def residual(p):\n        G=warp(p);v=(project(a,G)-project(w,G))/dt[:,None]\n        den=np.c_[np.r_[w,a],np.ones(len(w)*2)]@G[2]\n        return np.r_[((v-p[2:4])/speed).ravel(),p[:2]*.04,np.maximum(.25-den,0)*20]\n    fit=least_squares(residual,np.r_[0,0,median],bounds=([-1.5,-1.5,-20,-20],[1.5,1.5,20,20]),\n                      loss=\'soft_l1\',f_scale=.08,max_nfev=300)\n    G=warp(fit.x);den=np.c_[np.r_[w,a],np.ones(len(w)*2)]@G[2]\n    if np.min(den)<.25:return {\'available\':False,\'reason\':\'refined_mapping_pole\'}\n    refined=np.linalg.inv(normalizer)@G@normalizer@H\n    result=estimate_flow(features,refined)\n    if not result.get(\'available\'):return result\n    result.update(method=\'learned_flow_constant_motion_refinement\',\n        constant_speed_assumed=True,refinement_parameters=fit.x.tolist(),\n        metric_scale_constraint=\'Preserve learned-map Jacobian at support centre\',\n        original_mapping=H.tolist(),velocity_relative_rms=float(np.sqrt(np.mean(residual(fit.x)[:2*len(w)]**2))))\n    return result\n', 'lens_calibration.py': '"""Division-model radial self-calibration from one rigid translation\'s optical flow.\n\nUses a standard division lens model and an independently implemented robust\nepipole fit. It is not Fitzgibbon\'s two-view quadratic-eigenvalue algorithm.\n"""\nimport numpy as np\nfrom scipy.optimize import least_squares\nfrom calibration_candidates import select_rows,shared_vp_delay\nfrom refine_flow import flow_vanishing_point\n\n\ndef undistort_division(points,coefficient,center,scale):\n    p=(np.asarray(points)-center)/scale\n    return p/(1+coefficient*np.sum(p*p,axis=-1,keepdims=True))*scale+center\n\n\ndef distort_division(points,coefficient,center,scale):\n    p=(np.asarray(points)-center)/scale;r2=np.sum(p*p,axis=-1,keepdims=True)\n    return p*2/(1+np.sqrt(np.maximum(1-4*coefficient*r2,1e-8)))*scale+center\n\n\ndef fit_motion_lens(features,width,height):\n    source=[];dest=[]\n    for row in features.get(\'measurements\',[]):\n        p=np.asarray(row.get(\'matched_points_image\',[]));q=np.asarray(row.get(\'matched_next_points_image\',[]))\n        if len(p):\n            selected=np.unique(np.linspace(0,len(p)-1,min(30,len(p))).astype(int))\n            source.extend(p[selected]);dest.extend(q[selected])\n    if len(source)<100:return {\'available\':False,\'reason\':\'insufficient_visual_correspondences\'}\n    center=np.array([width/2,height/2]);scale=max(width,height)\n    a=(np.asarray(source)-center)/scale;b=(np.asarray(dest)-center)/scale\n    weights=np.clip(np.linalg.norm(b-a,axis=1)*scale,1,20)\n    initial_vp=(np.asarray(flow_vanishing_point(features)[\'point\'])-center)/scale\n    def convert(p,k):return p/np.maximum(1+k*np.sum(p*p,axis=1,keepdims=True),.1)\n    def residual(parameters):\n        vp=parameters[:2];k=parameters[2];p=convert(a,k);q=convert(b,k)\n        ray=p-vp;motion=q-p\n        cross=ray[:,0]*motion[:,1]-ray[:,1]*motion[:,0]\n        sine=cross/(np.linalg.norm(ray,axis=1)*np.linalg.norm(motion,axis=1)+1e-12)\n        den=np.r_[1+k*np.sum(a*a,axis=1),1+k*np.sum(b*b,axis=1)]\n        return np.r_[sine*weights,np.maximum(.25-den,0)*20,parameters[2]*.01]\n    fits=[]\n    for k in [-1,0,1]:\n        fit=least_squares(residual,np.r_[initial_vp,k],bounds=([-2,-2,-2],[2,2,2]),\n                          loss=\'soft_l1\',f_scale=.25,max_nfev=300)\n        score=float(np.mean(np.minimum(residual(fit.x)**2,1)));fits.append((score,fit))\n    score,fit=min(fits,key=lambda x:x[0]);k=float(fit.x[2])\n    singular=np.linalg.svd(fit.jac,compute_uv=False)\n    condition=float(singular[0]/max(singular[-1],1e-12))\n    available=abs(k)<1.98 and condition<1e5\n    return {\'available\':bool(available),\'division_coefficient\':k,\'center\':center.tolist(),\'scale\':scale,\n        \'vanishing_point_undistorted\':(fit.x[:2]*scale+center).tolist(),\n        \'correspondences\':len(a),\'robust_normal_flow_score\':score,\'jacobian_condition\':condition,\n        \'reason\':\'estimated_from_visual_motion\' if available else \'distortion_fit_at_bound_or_ill_conditioned\',\n        \'reference_label_used\':False}\n\n\ndef distortion_aware_delay(observations,report,features):\n    lens=fit_motion_lens(features,report[\'width\'],report[\'height\'])\n    if not lens[\'available\']:return {\'available\':False,\'reason\':lens[\'reason\'],\'lens_calibration\':lens}\n    corrected=[]\n    for row in observations:\n        points=np.asarray(row[\'keypoints\']).copy()\n        points[:,:2]=undistort_division(points[:,:2],lens[\'division_coefficient\'],lens[\'center\'],lens[\'scale\'])\n        corrected.append({**row,\'keypoints\':points.tolist()})\n    result=shared_vp_delay(select_rows(corrected,report),vanishing_point=lens[\'vanishing_point_undistorted\'],vp_weight=20)\n    result.update(method=\'distortion_aware_flow_vp_temporal_calibration\',lens_calibration=lens)\n    for key in [\'fitted_front_image\',\'fitted_rear_image\']:\n        if key in result:\n            result[key]=distort_division(result[key],lens[\'division_coefficient\'],lens[\'center\'],lens[\'scale\']).tolist()\n    return result\n', 'refinement_inference.py': '"""Second calibration-development round, using first-round learned maps and new flow."""\nimport argparse,json\nfrom pathlib import Path\nfrom benchmark_speed import load_clip\nfrom calibration_candidates import select_rows,shared_vp_delay,vp_rigid_bundle\nfrom lens_calibration import distortion_aware_delay\nfrom refine_flow import extract_adaptive_flow,refine_constant_motion,flow_vanishing_point\nfrom flow_calibration import estimate_flow\n\n\ndef run(args):\n    import cv2\n    cv2.setNumThreads(4)\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    for folder in sorted(Path(args.previous).iterdir()):\n        if not (folder/\'predictions.json\').exists():continue\n        target=out/folder.name;target.mkdir(exist_ok=True)\n        obs,report=load_clip(Path(args.inference_dir)/folder.name)\n        tracks=json.loads((Path(args.inference_dir)/folder.name/\'tracks.json\').read_text())\n        original=json.loads((folder/\'predictions.json\').read_text())\n        previous_flow=json.loads((folder/\'flow_features.json\').read_text())\n        path=target/\'adaptive_flow_features.json\'\n        flow=json.loads(path.read_text()) if path.exists() else None\n        if flow is None or not any(\'matched_points_image\' in r for r in flow.get(\'measurements\',[])):\n            flow=extract_adaptive_flow(Path(args.video_dir)/(folder.name+\'.MP4\'),obs,report,tracks)\n            path.write_text(json.dumps(flow,indent=2),encoding=\'utf-8\')\n        old_vp=flow_vanishing_point(previous_flow);vp=flow_vanishing_point(flow)\n        rows=select_rows(obs,report)\n        learned=original[\'v4_learned_mask\']\n        predictions={\'v2c_corner_vp_delay\':shared_vp_delay(rows),\n            \'v2d_flow_vp_delay\':shared_vp_delay(rows,vanishing_point=old_vp[\'point\'],vp_weight=20),\n            \'v5c_adaptive_learned_flow\':estimate_flow(flow,learned[\'mapping\']),\n            \'v5d_constant_motion_refinement\':refine_constant_motion(flow,learned[\'mapping\']),\n            \'v7_adaptive_vp_delay\':shared_vp_delay(rows,vanishing_point=vp[\'point\'],vp_weight=20),\n            \'v7b_adaptive_vp_rigid\':vp_rigid_bundle(obs,report,vp[\'point\']),\n            \'v7c_fitted_effective_length\':vp_rigid_bundle(obs,report,vp[\'point\'],fit_length=True),\n            \'v9_motion_lens_delay\':distortion_aware_delay(obs,report,flow)}\n        (target/\'predictions.json\').write_text(json.dumps(predictions,indent=2),encoding=\'utf-8\')\n        (target/\'vanishing_points.json\').write_text(json.dumps({\'original_flow\':old_vp,\'adaptive_flow\':vp},indent=2),encoding=\'utf-8\')\n        print(folder.name,{k:round(v[\'estimated_kmh\'],2) if v.get(\'estimated_kmh\') is not None else None for k,v in predictions.items()},flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in [\'inference-dir\',\'video-dir\',\'previous\',\'output\']:parser.add_argument(\'--\'+name,required=True)\n    run(parser.parse_args())\n'}
selected=['KiaSportage_100', 'KiaSportage_103', 'KiaSportage_33', 'KiaSportage_35', 'KiaSportage_38', 'KiaSportage_41', 'KiaSportage_46', 'KiaSportage_48', 'KiaSportage_51', 'KiaSportage_53', 'KiaSportage_58', 'KiaSportage_61', 'KiaSportage_63', 'KiaSportage_68', 'KiaSportage_69', 'KiaSportage_72', 'KiaSportage_74', 'KiaSportage_78', 'KiaSportage_80', 'KiaSportage_83', 'KiaSportage_86', 'KiaSportage_89', 'KiaSportage_91', 'KiaSportage_93', 'KiaSportage_98']
root=Path('/kaggle/working/pipeline');root.mkdir(exist_ok=True)
for name,source in sources.items():(root/name).write_text(source,encoding='utf-8')
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.175','lap==0.5.13'])
videos=Path('/kaggle/working/selected_videos');videos.mkdir(exist_ok=True)
for name in selected:
    matches=list(Path('/kaggle/input').rglob(name+'.MP4'));assert len(matches)==1,(name,matches)
    (videos/(name+'.MP4')).symlink_to(matches[0])
out=Path('/kaggle/working/kia_measurements');out.mkdir(exist_ok=True)
def run(script,*args):subprocess.check_call([sys.executable,str(root/script),*map(str,args)])
run('validation_inference.py','--input',videos,'--output',out/'inference','--device','0')
run('research_inference.py','--inference-dir',out/'inference','--video-dir',videos,
    '--output',out/'round1','--runtime','/kaggle/working/runtime/naver_cctv','--device','cuda')
run('refinement_inference.py','--inference-dir',out/'inference','--video-dir',videos,
    '--previous',out/'round1','--output',out/'round2')
import torch,torchvision,cv2
report={'videos':selected,'reference_labels_read':False,'text_annotations_read':False,
        'torch':torch.__version__,'torchvision':torchvision.__version__,'opencv':cv2.__version__,
        'gpu':torch.cuda.get_device_name(),
        'source_sha256':{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in sources}}
(out/'run_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('All speed-blind measurements complete:',len(selected))
